In [1]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
warnings.filterwarnings('ignore')

dataset_number = 12
split_ratios = [0.7, 0.8]
use_tuned = True
top_mode = 'global'
res_dir = '../Results/'

base_names = ['Random Forest', 'Logistic Regression', 'Decision Tree', 'XGBoost', 'CatBoost', 'AdaBoost', 'LightGBM', 'Support Vector Machine', 'K-Nearest Neighbors', 'Extra Trees', 'Gradient Boosting', 'Naive Bayes']

best_params = json.load(open(res_dir + 'best_params.json')) if os.path.exists(res_dir + 'best_params.json') else {}
top_info = json.load(open(res_dir + 'top_classifiers.json')) if os.path.exists(res_dir + 'top_classifiers.json') else {}

In [2]:
def make_model(name, params={}):
    models = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=3000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
        'CatBoost': CatBoostClassifier(verbose=0, allow_writing_files=False, random_state=42),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'LightGBM': LGBMClassifier(verbose=-1, random_state=42),
        'Support Vector Machine': SVC(probability=True, random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(),
        'Extra Trees': ExtraTreesClassifier(random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Naive Bayes': GaussianNB()
    }
    model = models[name]
    if use_tuned and len(params) > 0:
        model.set_params(**params)
    return model

In [3]:
def load_split(ratio):
    folder = '../Datasets/Split_' + str(round(ratio * 100)) + '/'
    train = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_train.csv')
    test = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_test.csv')
    return train.drop('label', axis=1), test.drop('label', axis=1), train['label'], test['label'], str(ratio)

In [4]:
accuracy_results = []
confusion_matrix_results = []
auc_results = []
training_testing_times = []

def record(ratio, name, y_test, pred, score, training_time, testing_time):
    accuracy_results.append({'Split Ratio': ratio, 'Classifier': name, 'Accuracy': accuracy_score(y_test, pred)})
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    confusion_matrix_results.append({
        'Split Ratio': ratio, 'Classifier': name, 'True Positive': tp, 'False Negative': fn, 'True Negative': tn, 'False Positive': fp,
        'True Positive Rate': tp / (tp + fn), 'False Negative Rate': fn / (tp + fn), 'True Negative Rate': tn / (tn + fp), 'False Positive Rate': fp / (tn + fp)})
    auc_results.append({'Split Ratio': ratio, 'Classifier': name, 'AUC Value': roc_auc_score(y_test, score)})
    training_testing_times.append({'Split Ratio': ratio, 'Classifier': name, 'Training Time (s)': training_time, 'Testing Time (s)': testing_time})

for ratio in split_ratios:
    X_train, X_test, y_train, y_test, key = load_split(ratio)
    tuned_params = best_params.get(key, {}).get('Dataset_' + str(dataset_number), {})
    fitted = {}
    probas = {}
    preds = {}
    train_times = {}

    for clf_name in base_names:
        clf = make_model(clf_name, tuned_params.get(clf_name, {}))
        start_time = time.time()
        clf.fit(X_train, y_train)
        training_time = time.time() - start_time
        start_time = time.time()
        pred = clf.predict(X_test)
        proba = clf.predict_proba(X_test)[:, 1]
        testing_time = time.time() - start_time
        fitted[clf_name] = clf
        probas[clf_name] = proba
        preds[clf_name] = pred
        train_times[clf_name] = training_time
        record(ratio, clf_name, y_test, pred, proba, training_time, testing_time)

    info = top_info.get(key, {}).get('global' if top_mode == 'global' else 'Dataset_' + str(dataset_number), {})
    for top in [3, 5, 7]:
        selected = info.get(str(top), base_names[:top])
        scores = info.get('scores', {})
        weights = np.array([scores.get(n, 1.0) for n in selected])

        stack = StackingClassifier(estimators=[(n, make_model(n, tuned_params.get(n, {}))) for n in selected], final_estimator=LogisticRegression(max_iter=3000), cv=5, n_jobs=-1)
        start_time = time.time()
        stack.fit(X_train, y_train)
        training_time = time.time() - start_time
        start_time = time.time()
        pred = stack.predict(X_test)
        proba = stack.predict_proba(X_test)[:, 1]
        testing_time = time.time() - start_time
        record(ratio, 'Stacking Top ' + str(top), y_test, pred, proba, training_time, testing_time)

        start_time = time.time()
        prob_matrix = np.column_stack([probas[n] for n in selected])
        pred_matrix = np.column_stack([preds[n] for n in selected])
        votes = {
            'Hard Voting': pred_matrix.mean(axis=1),
            'Soft Voting': prob_matrix.mean(axis=1),
            'Weighted Voting': np.average(prob_matrix, axis=1, weights=weights)
        }
        vote_time = time.time() - start_time
        base_train = sum(train_times[n] for n in selected)
        for vote_name, score in votes.items():
            record(ratio, vote_name + ' Top ' + str(top), y_test, (score >= 0.5).astype(int), score, base_train, vote_time)

accuracy_df = pd.DataFrame(accuracy_results)
confusion_matrix_df = pd.DataFrame(confusion_matrix_results)
auc_df = pd.DataFrame(auc_results)
training_testing_times_df = pd.DataFrame(training_testing_times)

In [5]:
print("Accuracy:")
accuracy_df

Accuracy:


,Split Ratio,Classifier,Accuracy
0,0.7,Random Forest,0.757282
1,0.7,Logistic Regression,0.757282
2,0.7,Decision Tree,0.689320
3,0.7,XGBoost,0.728155
4,0.7,CatBoost,0.786408
5,0.7,AdaBoost,0.718447
6,0.7,LightGBM,0.776699
7,0.7,Support Vector Machine,0.757282
8,0.7,K-Nearest Neighbors,0.660194
9,0.7,Extra Trees,0.718447


In [6]:
print("\nConfusion Matrix Metrics:")
confusion_matrix_df


Confusion Matrix Metrics:


,Split Ratio,Classifier,True Positive,False Negative,True Negative,False Positive,True Positive Rate,False Negative Rate,True Negative Rate,False Positive Rate
0,0.7,Random Forest,25,13,53,12,0.657895,0.342105,0.815385,0.184615
1,0.7,Logistic Regression,24,14,54,11,0.631579,0.368421,0.830769,0.169231
2,0.7,Decision Tree,29,9,42,23,0.763158,0.236842,0.646154,0.353846
3,0.7,XGBoost,27,11,48,17,0.710526,0.289474,0.738462,0.261538
4,0.7,CatBoost,28,10,53,12,0.736842,0.263158,0.815385,0.184615
5,0.7,AdaBoost,25,13,49,16,0.657895,0.342105,0.753846,0.246154
6,0.7,LightGBM,27,11,53,12,0.710526,0.289474,0.815385,0.184615
7,0.7,Support Vector Machine,22,16,56,9,0.578947,0.421053,0.861538,0.138462
8,0.7,K-Nearest Neighbors,14,24,54,11,0.368421,0.631579,0.830769,0.169231
9,0.7,Extra Trees,22,16,52,13,0.578947,0.421053,0.800000,0.200000


In [7]:
print("\nAUC Values:")
auc_df


AUC Values:


,Split Ratio,Classifier,AUC Value
0,0.7,Random Forest,0.829150
1,0.7,Logistic Regression,0.778947
2,0.7,Decision Tree,0.704656
3,0.7,XGBoost,0.811741
4,0.7,CatBoost,0.841296
5,0.7,AdaBoost,0.812348
6,0.7,LightGBM,0.837247
7,0.7,Support Vector Machine,0.762348
8,0.7,K-Nearest Neighbors,0.687652
9,0.7,Extra Trees,0.756883


In [8]:
os.makedirs(res_dir, exist_ok=True)
merged = accuracy_df.merge(auc_df, on=['Split Ratio', 'Classifier']).merge(confusion_matrix_df, on=['Split Ratio', 'Classifier']).merge(training_testing_times_df, on=['Split Ratio', 'Classifier'])
merged.to_csv(res_dir + 'Split_Dataset_' + str(dataset_number) + '.csv', index=False)
merged.sort_values('AUC Value', ascending=False).head(10)

,Split Ratio,Classifier,Accuracy,AUC Value,True Positive,False Negative,True Negative,False Positive,True Positive Rate,False Negative Rate,True Negative Rate,False Positive Rate,Training Time (s),Testing Time (s)
34,0.8,Gradient Boosting,0.797101,0.877460,21,5,34,9,0.807692,0.192308,0.790698,0.209302,0.355839,0.009362
20,0.7,Stacking Top 7,0.786408,0.851822,27,11,54,11,0.710526,0.289474,0.830769,0.169231,6.804189,0.132854
24,0.8,Random Forest,0.811594,0.849732,19,7,37,6,0.730769,0.269231,0.860465,0.139535,0.144984,0.016014
10,0.7,Gradient Boosting,0.766990,0.848583,27,11,52,13,0.710526,0.289474,0.800000,0.200000,0.077969,0.000000
47,0.8,Weighted Voting Top 7,0.768116,0.847048,20,6,33,10,0.769231,0.230769,0.767442,0.232558,1.287504,0.000000
44,0.8,Stacking Top 7,0.768116,0.847048,20,6,33,10,0.769231,0.230769,0.767442,0.232558,3.169173,0.128191
46,0.8,Soft Voting Top 7,0.782609,0.846154,20,6,34,9,0.769231,0.230769,0.790698,0.209302,1.287504,0.000000
23,0.7,Weighted Voting Top 7,0.757282,0.844130,27,11,51,14,0.710526,0.289474,0.784615,0.215385,3.296045,0.000000
22,0.7,Soft Voting Top 7,0.757282,0.844130,27,11,51,14,0.710526,0.289474,0.784615,0.215385,3.296045,0.000000
16,0.7,Stacking Top 5,0.776699,0.842105,27,11,53,12,0.710526,0.289474,0.815385,0.184615,12.570280,0.111619
